*0.1 Python for GenAI*

# Logging

**The situation.** At 3 a.m. a customer reports wrong answers from the last hour. The service's only output is `print` statements: no timestamps, no severity, nothing saying which part of the program wrote a line — and 90% of the lines are `HTTP Request: POST …` from the SDK.

**The fix: real log lines, and library noise turned down.** The `logging` module writes lines with a time, a *level* (INFO for normal events, WARNING for odd-but-handled, ERROR for failures) and the name of the logger that wrote it. And it lets you set each library's level separately, so the SDK's chatter disappears.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**A logger for the chat service.** Configured once. Then a real model call, logged before and after.

In [2]:
import io
import logging
import time

from openai import OpenAI

buffer = io.StringIO()  # in a service this would be stdout; here it is captured to read back
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-7s %(name)s: %(message)s",
    stream=buffer,
    force=True,
)
log = logging.getLogger("support.chat")

client = OpenAI(timeout=30)
started = time.perf_counter()
log.info("completion.start model=%s", MODEL)
reply = client.chat.completions.create(
    model=MODEL, messages=[{"role": "user", "content": "Say hi in one word."}], temperature=0
)
log.info(
    "completion.end seconds=%.2f tokens=%d", time.perf_counter() - started, reply.usage.total_tokens
)
log.warning("budget.near_limit spent_usd=%.4f limit_usd=%.4f", 0.048, 0.05)
log.debug("hidden: DEBUG is below the INFO level")
print(buffer.getvalue())
assert "completion.end" in buffer.getvalue()

2026-09-21 11:50:49,743 INFO    support.chat: completion.start model=gpt-4o-mini
2026-09-21 11:50:50,533 INFO    httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-21 11:50:50,546 INFO    support.chat: completion.end seconds=0.80 tokens=15
2026-09-21 11:50:50,546 WARNING support.chat: budget.near_limit spent_usd=0.0480 limit_usd=0.0500



**Reading the output.** Three lines you wrote, each with time, level and logger name — plus one you did not: the SDK's `HTTP Request` line, under the logger name `httpx`, the HTTP library the SDK uses. Every library logs under its own name, and that name is the handle you turn.

**Turn the noise down.** Set that logger to WARNING and make another call.

In [3]:
logging.getLogger("httpx").setLevel(logging.WARNING)  # the SDK's HTTP client
buffer.truncate(0)
buffer.seek(0)
client.chat.completions.create(
    model=MODEL, messages=[{"role": "user", "content": "Say hi again."}], temperature=0
)
log.info("completion.end second call, HTTP chatter silenced")
print(buffer.getvalue())
assert "HTTP Request" not in buffer.getvalue()

2026-09-21 11:50:51,126 INFO    support.chat: completion.end second call, HTTP chatter silenced



**The rule to remember.** Configure logging once at start-up, at INFO in production; set every library to WARNING; never log keys or whole prompts by default.

| Use it when | Don't when | Instead use |
|---|---|---|
| every service and job | notebooks and one-off scripts — `print` is fine | structlog (next item) when logs go to a search system |

**Watch out**
- Silence libraries by their exact logger name (`httpx`, `openai`, `urllib3`); a near-miss changes nothing.
- Write to stdout and let the platform collect it; files inside a container disappear with the container.
- Levels must mean something: if everything is ERROR, alerts are noise.